# Demo - Train a Subtype Classifier on the KiTS23 Data
[KiTS23](https://kits-challenge.org/kits23/) was a competition were teams competed to develop systems for segmentation of kidneys, tumors and cysts.
You can download their dataset [here](https://github.com/neheller/kits23). 

But worry not, we already extracted the features and saved them in [data/kits_embeddings.parquet](../data/kits_radiomics.parquet).


## Feature Handling

### 1. Extract Features
Refer to [Demo_binary.ipynb](Demo_binary.ipynb)

### 2. Define New Classes
We extracte features for 0:tumors and 1:cysts. But what if we want classify the specifc tumor subtype? Luckily this information can be found in the KITS metadata.

Let's define six tumor subtypes that we are interested in:

In [ ]:
import pandas as pd

features = pd.read_parquet("../data/kits_embeddings.parquet")
metadata = pd.read_csv("../data/KITS.csv")


class_dict = {
    "clear_cell_rcc":0,
    "papillary_rcc":1,
    "chromophobe_rcc":2,
    "oncocytoma":3,
    "transitional_cell_carcinoma":5,
    "clear_cell_papillary":5,
    "other":5,
    "angiomyolipoma":5,
    "rcc_unclassified":5,
    "mest":5,
    "wilms_tumor":5,
    "spindle_cell_neoplasm":5,
    "multilocular_cystic_rcc":5,
    "cyst":4,
}

names_multiclass = {
    0: "ccRCC",
    1: "pRCC",
    2: "chrRCC",
    3: "Oncocytoma",
    4: "Cyst",
    5: "Other"
}

metadata["new_class_id"] = metadata["subtype"].map(class_dict)
metadata["subtype"] = metadata["new_class_id"].map(names_multiclass)
metadata["subtype"].value_counts()

Now we need to reassing our feature classes. All features for solid lesions get one of the newly defined class_ids. All features of cysts will be remapped to 4:cyst.

In [ ]:
def map_class(row):
    if row["class_id"] == 0:
        return row["new_class_id"]
    else:
        return 4  # cyst
    
multiclass_features = pd.merge(features, metadata[["case", "new_class_id"]], on="case")
multiclass_features["class_id"] = multiclass_features.apply(map_class, axis=1)
multiclass_features = multiclass_features.drop(columns=["new_class_id"])
multiclass_features = multiclass_features[~multiclass_features["class_id"].isna()]
multiclass_features["class_id"] = multiclass_features["class_id"].astype(int)

### 2. Inspect Data

In [ ]:
def describe_data(features):
    n_cases = features['case'].nunique()
    n_lesions = len(features[~features['augmented']])
    print(f"Extracted features for {n_lesions} lesions from {n_cases} cases.")

    classes = features['class_id'].unique()
    print(f"Found {len(classes)} classes: {classes}")

    for cl in classes:
        n_cl_lesions = len(features[(features['class_id'] == cl) & (~features['augmented'])])
        print(f"  Class {cl}: {n_cl_lesions} lesions")

    oversampling_factor = (len(features)-n_lesions) / n_lesions 
    print(f"Each lesion was augmented {oversampling_factor:.1f} times on average.")

describe_data(multiclass_features)

### 3. Split Data
Now that we have features we can split them into a train and a test partition. We also remove all augmentations (if any) from the test partition

In [ ]:

from renal_vision.shared.utils import generate_stratified_group_split

train, test = generate_stratified_group_split(multiclass_features,group_col="case")
test = test[~test['augmented']].reset_index(drop=True)

train.to_parquet("multiclass_features_train.parquet", index=False)
test.to_parquet("multiclass_features_test.parquet", index=False)

print("Train set:")
describe_data(train)
print("\nTest set:")
describe_data(test)

## Training

### 1. [Optional] Generate Task Specifications
Lets specify some meta data that helps us to understand the models output better.

In [ ]:
import json

with open("names_multiclass.json","w") as f:
    json.dump(names_multiclass, f)    

### 2. Train Models

In [ ]:
! rv train \
    --data multiclass_features_train.parquet \
    --model xgboost \
    --extractor-config multiclass_features_train.config.json \
    --output-dir multiclass \
    --class-config "names_multiclass.json"

## Evaluate Models

In [ ]:
# jupyter IPython may throw an error when plotting the confusion matrix.
# If that happens, please run the following command in a terminal instead.
! rv eval \
    --data multiclass_features_test.parquet \
    --model multiclass/model.pkl \
    --output-dir test